<a href="https://colab.research.google.com/github/ZIZOUTOUMOU/LAB02-AdvDB/blob/main/algeria_wilayas_to_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Population of Algeria and of every wilaya → one CSV file

- **Worldometer** has no data per wilaya, so the wilaya figures come from the official census tables on Wikipedia (ONS census 1987, 1998, 2008).
- Algeria's latest total comes from **worldometers.info**.
- Output: **`algeria_wilayas_population.csv`**: first row = Algeria, then one row per wilaya.

**Runtime → Run all**

In [1]:
import re
import requests
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                         "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"}

def fetch(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    r.encoding = "utf-8"
    return r.text

def to_int(text):
    """'2,988,145[1]' -> 2988145   |   '557 906 ' -> 557906"""
    text = re.sub(r"\[.*?\]", "", text or "")
    digits = re.sub(r"[^\d]", "", text)
    return int(digits) if digits else None

def row_texts(tr):
    return [c.get_text(" ", strip=True) for c in tr.find_all(["td", "th"])]

In [2]:
# ---- Population of every wilaya (official census 1987 / 1998 / 2008) ----
soup = BeautifulSoup(fetch("https://en.wikipedia.org/wiki/List_of_Algerian_provinces_by_population"), "html.parser")
table = next(t for t in soup.select("table") if "2008 census" in t.get_text())

wilayas, country_census = [], None
for tr in table.select("tr"):
    texts = row_texts(tr)
    if not texts:
        continue
    if texts[0].lower() == "total":                       # last row = whole of Algeria
        nums = [to_int(x) for x in texts[1:] if to_int(x)]
        country_census = nums[-3:]
    elif len(texts) >= 5 and to_int(texts[0]) is not None:
        wilayas.append({
            "wilaya_code": to_int(texts[0]),
            "wilaya": re.sub(r"\s*Province$", "", texts[1]),
            "pop_1987": to_int(texts[2]),
            "pop_1998": to_int(texts[3]),
            "pop_2008": to_int(texts[4]),
        })

pop_df = pd.DataFrame(wilayas)
print(len(pop_df), "wilayas found | Algeria total (1987, 1998, 2008):", country_census)
pop_df.head()

48 wilayas found | Algeria total (1987, 1998, 2008): [22971658, 29276767, 34080030]


,wilaya_code,wilaya,pop_1987,pop_1998,pop_2008
0,16,Algiers,1687579,2561992,2988145
1,31,Oran,916678,1155464,1584607
2,19,Sétif,997482,1315940,1496150
3,17,Djelfa,490240,860981,1223223
4,5,Batna,757059,968820,1128030


In [3]:
# ---- Area of every wilaya ----
soup = BeautifulSoup(fetch("https://en.wikipedia.org/wiki/List_of_Algerian_provinces_by_area"), "html.parser")
table = next(t for t in soup.select("table") if "Number on map" in t.get_text())

areas, country_area = [], None
for tr in table.select("tr"):
    texts = row_texts(tr)
    if not texts:
        continue
    if texts[0].lower() == "total":
        country_area = next((to_int(x.split("km")[0]) for x in texts if "km" in x), None)
    elif len(texts) >= 5 and to_int(texts[1]) is not None:
        areas.append({"wilaya_code": to_int(texts[1]), "area_km2": to_int(texts[3].split("km")[0])})

area_df = pd.DataFrame(areas)
print(len(area_df), "areas found | Algeria total area:", country_area, "km2")

48 areas found | Algeria total area: 2381741 km2


In [4]:
# ---- Latest estimate for the whole country from worldometers.info (optional) ----
latest_pop = latest_year = None
try:
    tables = pd.read_html(StringIO(fetch("https://www.worldometers.info/world-population/algeria-population/")))
    for t in tables:
        if "Year" in t.columns and "Population" in t.columns and t["Year"].astype(int).min() <= 1960:
            top = t.sort_values("Year").iloc[-1]
            latest_year, latest_pop = int(top["Year"]), to_int(str(top["Population"]))
    print("Worldometer:", latest_year, "->", f"{latest_pop:,}")
except Exception as e:
    print("Worldometer skipped:", e)

Worldometer: 2026 -> 48,028,334


In [5]:
# ---- Build the final table: Algeria on the first row, then every wilaya ----
df = pop_df.merge(area_df, on="wilaya_code", how="left")

country = {
    "level": "country", "wilaya_code": None, "wilaya": "Algeria",
    "pop_1987": country_census[0], "pop_1998": country_census[1], "pop_2008": country_census[2],
    "area_km2": country_area,
    "latest_population": latest_pop, "latest_population_year": latest_year,
}
df["level"] = "wilaya"
df = pd.concat([pd.DataFrame([country]), df.sort_values("wilaya_code")], ignore_index=True)

national_2008 = country_census[2]
df["density_2008_per_km2"] = (df.pop_2008 / df.area_km2).round(1)
df["share_of_national_2008_pct"] = (df.pop_2008 / national_2008 * 100).round(2)
df["growth_1987_1998_pct"] = ((df.pop_1998 / df.pop_1987 - 1) * 100).round(2)
df["growth_1998_2008_pct"] = ((df.pop_2008 / df.pop_1998 - 1) * 100).round(2)
df["annual_growth_1998_2008_pct"] = (((df.pop_2008 / df.pop_1998) ** (1 / 10) - 1) * 100).round(2)

df = df[["level", "wilaya_code", "wilaya", "pop_1987", "pop_1998", "pop_2008", "area_km2",
         "density_2008_per_km2", "share_of_national_2008_pct", "growth_1987_1998_pct",
         "growth_1998_2008_pct", "annual_growth_1998_2008_pct",
         "latest_population", "latest_population_year"]]

df.to_csv("algeria_wilayas_population.csv", index=False, encoding="utf-8-sig")
print("Saved algeria_wilayas_population.csv:", len(df), "rows (1 country + wilayas),", len(df.columns), "columns")
df

Saved algeria_wilayas_population.csv: 49 rows (1 country + wilayas), 14 columns


,level,wilaya_code,wilaya,pop_1987,pop_1998,pop_2008,area_km2,density_2008_per_km2,share_of_national_2008_pct,growth_1987_1998_pct,growth_1998_2008_pct,annual_growth_1998_2008_pct,latest_population,latest_population_year
0,country,None,Algeria,22971658,29276767,34080030,2381741,14.3,100.00,27.45,16.41,1.53,48028334.0,2026.0
1,wilaya,1,Adrar,216931,311952,439693,427368,1.0,1.29,43.80,40.95,3.49,NaN,NaN
2,wilaya,2,Chlef,679717,884978,1013718,4791,211.6,2.97,30.20,14.55,1.37,NaN,NaN
3,wilaya,3,Laghouat,215183,327634,477328,25057,19.0,1.40,52.26,45.69,3.83,NaN,NaN
4,wilaya,4,Oum El Bouaghi,402683,533711,644364,7638,84.4,1.89,32.54,20.73,1.90,NaN,NaN
5,wilaya,5,Batna,757059,968820,1128030,12192,92.5,3.31,27.97,16.43,1.53,NaN,NaN
6,wilaya,6,Béjaïa,697669,848560,915835,3268,280.2,2.69,21.63,7.93,0.77,NaN,NaN
7,wilaya,7,Biskra,429217,588648,730262,20986,34.8,2.14,37.14,24.06,2.18,NaN,NaN
8,wilaya,8,Béchar,183896,230482,274866,162200,1.7,0.81,25.33,19.26,1.78,NaN,NaN
9,wilaya,9,Blida,704462,787069,1009892,1478,683.3,2.96,11.73,28.31,2.52,NaN,NaN


In [6]:
from google.colab import files
files.download("algeria_wilayas_population.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>